# How to Use Granite Embedding Models

In [ ]:
import warnings

warnings.filterwarnings('ignore')

## Use IBM granite embedding model with SentenceTransformer

This notebook demonstrates how to use IBM Granite Embedding Models for text embedding tasks. The notebook covers:

1. **Model Usage**: Two approaches to use Granite embedding models
   - Using Sentence Transformers library (recommended for ease of use)
   - Using Hugging Face Transformers library (for more control)

2. **Fine-tuning**: How to fine-tune the model on custom data using MS MARCO dataset

### Key Features
- **Model**: `ibm-granite/granite-embedding-english-r2` - English text embedding model
- **Device**: CUDA-enabled GPU support or CPU (change the value of device to `cpu` for CPU)
- **Normalization**: Optional embedding normalization for downstream tasks
- **Pooling Strategy**: CLS token pooling for text representation


In [ ]:
from sentence_transformers import SentenceTransformer, util

device="cuda"

model_path = "ibm-granite/granite-embedding-english-r2"
# Load the Sentence Transformer model
model = SentenceTransformer(model_path, device=device)

input_queries = [
    ' Who made the song My achy breaky heart? ',
    'summit define'
    ]

input_passages = [
    "Achy Breaky Heart is a country song written by Don Von Tress. Originally titled Don't Tell My Heart and performed by The Marcy Brothers in 1991. ",
    "Definition of summit for English Language Learners. : 1 the highest point of a mountain : the top of a mountain. : 2 the highest level. : 3 a meeting or series of meetings between the leaders of two or more governments."
    ]

# encode queries and passages. The model produces unnormalized vectors. If your task requires normalized embeddings pass normalize_embeddings=True to encode as below.
query_embeddings = model.encode(input_queries)
passage_embeddings = model.encode(input_passages)

# calculate cosine similarity
print(util.cos_sim(query_embeddings, passage_embeddings))

### Use our models with Huggingface Transformers

This section demonstrates how to use the IBM Granite embedding model directly with the HuggingFace Transformers library directly. This approach provides more control over the tokenization and embedding generation process compared to using Sentence Transformers.

**Key steps covered:**
- Loading the model and tokenizer using `AutoModel` and `AutoTokenizer`
- Manual tokenization with padding and truncation
- Generating embeddings using the model's forward pass
- Extracting CLS token embeddings for sequence representation
- Normalizing embeddings for similarity tasks

This method is useful when you need fine-grained control over the embedding process or want to integrate the model into custom pipelines.


In [ ]:
# Import required libraries for PyTorch-based embedding generation
import torch
from transformers import AutoModel, AutoTokenizer

# Specify the Granite embedding model from IBM
model_path = "ibm-granite/granite-embedding-english-r2"

# Load the pre-trained model and tokenizer
# The model is moved to the specified device (GPU/CPU) for computation
model = AutoModel.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
# Set model to evaluation mode to disable dropout and batch normalization updates
model.eval()

# Define sample input queries for embedding generation
# These are example texts that will be converted to numerical embeddings
input_queries = [
    ' Who made the song My achy breaky heart? ',  # Question about song authorship
    'summit define'  # Request for definition of "summit"
]

# Tokenize the input queries
# - padding=True: Pad shorter sequences to match the longest sequence in the batch
# - truncation=True: Truncate sequences that exceed the model's maximum length
# - return_tensors='pt': Return PyTorch tensors instead of lists
# - .to(device): Move tensors to the same device as the model (GPU/CPU)
tokenized_queries = tokenizer(input_queries, padding=True, truncation=True, return_tensors='pt').to(device)

# Generate embeddings from the tokenized inputs
with torch.no_grad():  # Disable gradient computation for inference (saves memory and computation)
    # Pass tokenized inputs through the model
    model_output = model(**tokenized_queries)
    # Extract embeddings using CLS pooling strategy
    # model_output[0] contains the hidden states, [:, 0] selects the CLS token (first token)
    # The CLS token represents the entire sequence in transformer models
    query_embeddings = model_output[0][:, 0]

# Normalize the embeddings to unit vectors
# This is important for downstream tasks like similarity computation
# L2 normalization ensures all embeddings have the same magnitude
query_embeddings = torch.nn.functional.normalize(query_embeddings, dim=1)


In [ ]:
query_embeddings

## Fine-tune our model on your own data

This section demonstrates how to fine-tune the IBM Granite embedding model on custom data using the MS MARCO dataset. The fine-tuning process includes:

1. **Dataset Preparation**: Loading and splitting the MS MARCO Co-Condenser dataset for training and evaluation
2. **Model Setup**: Configuring the Granite embedding model with appropriate training parameters
3. **Training Configuration**: Setting up loss functions, training arguments, and evaluation metrics
4. **Fine-tuning Process**: Training the model with cached multiple negatives ranking loss
5. **Evaluation**: Assessing model performance using triplet evaluation on the development set

**Key components:**
- **Dataset**: MS MARCO Co-Condenser triplet dataset (query-positive-negative pairs)
- **Loss Function**: CachedMultipleNegativesRankingLoss for contrastive learning
- **Evaluation**: TripletEvaluator to measure embedding quality
- **Training**: 1 epoch with learning rate 2e-5 and mixed precision training


In [ ]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import TripletEvaluator
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [ ]:
# Create the model and set hyperparameters
lr = 2e-5
model_name = "ibm-granite/granite-embedding-english-r2"
output_dir = f"granite-r2-msmarco-{lr}"

model = SentenceTransformer(model_name)

In [ ]:
# Load the dataset
dataset = load_dataset(
        "sentence-transformers/msmarco-co-condenser-margin-mse-sym-mnrl-mean-v1",
        "triplet-hard",
        split="train",
    )
dataset_dict = dataset.train_test_split(test_size=2_000, seed=43)
train_dataset = dataset_dict["train"].select(range(500))
eval_dataset = dataset_dict["test"]

In [ ]:
# Create training hyperparameters
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=512)  # Increase mini_batch_size if you have enough VRAM

args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=512,
    warmup_ratio=0.05,
    fp16=False,  # Set to False if GPU can't handle FP16
    bf16=True,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # (Cached)MultipleNegativesRankingLoss benefits from no duplicates
    learning_rate=lr,
    # Optional tracking/debugging parameters:
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    report_to="none"
)

In [ ]:
# Create the dev evaluator to use in training
dev_evaluator = TripletEvaluator(
    anchors=eval_dataset["query"],
    positives=eval_dataset["positive"],
    negatives=eval_dataset["negative"],
    name="msmarco-co-condenser-dev",
    )
dev_evaluator(model)

In [ ]:
# Train the fine-tuned Granite embedding model
# This cell executes the actual training process using the configured trainer

# The SentenceTransformerTrainer combines all previously configured components:
# - model: The Granite embedding model to be fine-tuned
# - args: Training hyperparameters (learning rate, batch size, epochs, etc.)
# - train_dataset: MS MARCO training data (500 samples for demonstration)
# - eval_dataset: MS MARCO evaluation data (2000 samples)
# - loss: CachedMultipleNegativesRankingLoss for contrastive learning
# - evaluator: TripletEvaluator for measuring embedding quality during training

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)

# Execute the training process
# This will run for 1 epoch with mixed precision training (BF16)
# Progress will be logged every 100 steps, model will be saved every 500 steps
trainer.train()


In [ ]:
# Evaluate the fine-tuned model performance
# This runs the TripletEvaluator on the current model to measure embedding quality/
# The evaluator uses the MS MARCO development set with query-positive-negative triplets.
# It calculates metrics like accuracy and ranking performance to assess how well
# the model can distinguish between relevant and irrelevant passages for given queries.

dev_evaluator(model)


In [ ]:
# Save the fine-tuned model
model.save_pretrained(f"{output_dir}/final")